#### MILE -- pSMILE Mini-Batch (02c), empfohlene SG-MCMC-Variante (statt SGHMC)

Gleicher Grundaufbau wie `02b_mile_sgld_test.ipynb` (Baseline laden, Grad-Estimator, Bypass-Trick
fuer pmap), aber mit dem eigentlich empfohlenen Kernel statt des SGHMC-Fallbacks:

- **Kernel: `NOISYSMILE`** aus `src/kernels/smile.py` im SMILE-Repo -- das ist die Preconditioned/
  Noise-korrigierte Ausbaustufe der *eigenen* Methode des Papers (SMILE -> SMILET -> NOISYSMILE),
  nutzt dieselbe isokinetische/mikrokanonische Dynamik wie unser Full-Batch-MCLMC (MILE), nur mit
  Mini-Batches. Das ist das, was in der Mail als "pSMILE" bezeichnet wurde -- nicht SGHMC (das war
  nur "wuerde auch funktionieren").
- **Eingebaute automatische Step-Size-Anpassung** ueber Energy-Error-Monitoring (Wilson-Hilferty-
  Gamma-Quantil-Approximation) -- adressiert die Kritik "Energy Variance nicht austariert".
  WICHTIG (aus dem Quellcode verifiziert, nicht geraten): `_sample_step` bekommt zwar einen
  `step_size`-Parameter, der wird aber im Funktionskoerper **nie benutzt** -- massgeblich ist einzig
  `state.step_size` (ein Array, PRO PARAMETER-DIMENSION, nicht ein globaler Skalar). Die Anfangs-
  Step-Size wird deshalb beim Bauen des States gesetzt (`NOISYSMILE.State(..., step_size=X)`), nicht
  beim Aufruf von `_sample_step`. Ab dem 10. `_sample_step`-Aufruf (state.time >= 10) passt sich
  `state.step_size` automatisch an; die ersten ~9 Aufrufe sind bewusst deaktiviert (Cold-Start-Schutz
  der Bibliothek selbst, NICHT von uns nachgebaut -- die Default-Initialwerte fuer
  `energy_var_mean=0.0`/`energy_var_std=0.0`/`time=1.0` sollten deshalb unveraendert von der
  Bibliothek uebernommen werden, nicht selbst vorbelegt werden).
- **Preconditioning**: `elementwise_sd` wird laufend aus der Gradienten-Varianz geschaetzt und
  normalisiert die Schritte pro Dimension -- das soll genau das "zu heisse" Laufverhalten
  korrigieren, das wir beim naiven SGHMC beobachtet haben (Distanz ~15-20 statt ~4,5-6,5 wie MCLMC).
- **Prior: `PRIOR_STD=1.0`** (Autoren-Empfehlung aus der Mail) als Default -- anders als beim
  naiven SGHMC ist hier eine begruendete Vermutung, dass das mit der Rauschkorrektur besser
  funktioniert als ohne. `PRIOR_STD=0.02` (unsere eigene Kalibrierung) bleibt als Fallback per
  Env-Var moeglich, falls 1.0 wieder unkontrolliert diffundiert.

**Erst ganz klein testen** (wie bei 02b): Defaults unten sind bewusst winzig. Der `jax.jit`-Fix
(einmalige Kompilierung ueber `functools.partial`, siehe 02b-Vorgeschichte) ist von Anfang an mit
eingebaut, nicht nachtraeglich.


In [1]:
from pathlib import Path
import os
import json as _json

# ============================================================
# CONFIG (02c -- pSMILE/NOISYSMILE Mini-Batch-Test)
# ============================================================

MASTER_DIR = Path("/dss/dsshome1/00/ra58vit2/Masterarbeit")
QWEN_REPO = MASTER_DIR / "bayes_sub_inf"
SMILE_REPO = MASTER_DIR / "SMILE"   # sollte durch 02b schon existieren

BASELINE_SCRIPT = (
    QWEN_REPO / "experiments" / "ag_news_qwen_lora" / "evaluate_saved_baseline.py"
)
RESULT_DIR = MASTER_DIR / "method_results" / "mile_qwen_agnews"
CHECKPOINTS_ROOT = Path.home() / "Masterarbeit" / "checkpoints"

CHAIN_SEEDS = _json.loads(os.environ.get("CHAIN_SEEDS", "[1]"))

SEQ_LEN = 32
PRIOR_STD = float(os.environ.get("PRIOR_STD", "1.0"))   # Autoren-Empfehlung: Standard-Gaussian

MINIBATCH_SIZE = int(os.environ.get("MINIBATCH_SIZE", "32"))
NLL_INNER_BATCH_SIZE = 16
assert MINIBATCH_SIZE % NLL_INNER_BATCH_SIZE == 0, (
    f"MINIBATCH_SIZE ({MINIBATCH_SIZE}) muss durch NLL_INNER_BATCH_SIZE ({NLL_INNER_BATCH_SIZE}) teilbar sein."
)

# Anfangs-Step-Size -- das ist der EINZIGE Hyperparameter, den die Bibliothek selbst zum Tunen
# vorschlaegt (alles andere bleibt auf Library-Default). Aus dem Quellcode verifiziert: dieser Wert
# geht in NOISYSMILE.State(step_size=...) beim Bauen des States, NICHT in _sample_step().
STEP_SIZE_GRID = _json.loads(os.environ.get("STEP_SIZE_GRID", "[0.001, 0.003, 0.01, 0.03, 0.1]"))

# Library-Defaults fuer die Energy-Error-Adaption -- bewusst NICHT veraendert (SMILE-Paper-Tuning-
# Rezept: alles auf Default lassen ausser der Step Size).
NUM_INTEGRATION_STEPS = int(os.environ.get("NUM_INTEGRATION_STEPS", "1"))
RESET_PROB = float(os.environ.get("RESET_PROB", "0.99"))
ADAPTION_PROB = float(os.environ.get("ADAPTION_PROB", "0.05"))
ADAPTION_STRENGTH = float(os.environ.get("ADAPTION_STRENGTH", "0.02"))
SMOOTHING_FACTOR = float(os.environ.get("SMOOTHING_FACTOR", "0.01"))

# GANZ KLEIN fuer den ersten Smoke-Test.
N_SAMPLES_PER_CHAIN = int(os.environ.get("N_SAMPLES_PER_CHAIN", "3"))
THINNING_STEPS = int(os.environ.get("THINNING_STEPS", "5"))

# Informeller Burn-in-Ersatz (NOISYSMILE hat keine separate Warmup-Phase):
# verwirft die ersten N Samples PRO CHAIN vor dem Kombinieren/Speichern.
# Aus den Testlaeufen (60 Samples/Chain, step_size=0.01, PRIOR_STD=0.02):
# alle 3 Chains erreichen ihre jeweilige Plateau-Region nach ca. 10-20 Samples.
BURN_IN_SAMPLES = int(os.environ.get("BURN_IN_SAMPLES", "0"))

EVAL_BATCH_SIZE = 64
EVAL_MODE = "test"

N_CHAINS = len(CHAIN_SEEDS)

RUN_NAME = (
    f"mile_psmile_k{N_CHAINS}"
    f"_prior{PRIOR_STD}"
    f"_mb{MINIBATCH_SIZE}"
    f"_spc{N_SAMPLES_PER_CHAIN}"
    f"_thin{THINNING_STEPS}"
    f"_{EVAL_MODE}eval"
)

print("======================================")
print("MILE -- pSMILE/NOISYSMILE MINI-BATCH-TEST (02c)")
print("======================================")
print("Run name:              ", RUN_NAME)
print("Chain seeds:           ", CHAIN_SEEDS)
print("Prior std:             ", PRIOR_STD, " (Autoren-Default: Standard-Gaussian)")
print("Minibatch size:        ", MINIBATCH_SIZE)
print("Initial step-size grid:", STEP_SIZE_GRID)
print("num_integration_steps: ", NUM_INTEGRATION_STEPS)
print("reset_prob:            ", RESET_PROB, " (Library-Default)")
print("adaption_prob:         ", ADAPTION_PROB, " (Library-Default)")
print("adaption_strength:     ", ADAPTION_STRENGTH, " (Library-Default)")
print("smoothing_factor:      ", SMOOTHING_FACTOR, " (Library-Default)")
print("N_SAMPLES_PER_CHAIN:   ", N_SAMPLES_PER_CHAIN, " (KLEIN -- Smoke-Test)")
print("THINNING_STEPS:        ", THINNING_STEPS, " (KLEIN -- Smoke-Test)")
print("BURN_IN_SAMPLES:       ", BURN_IN_SAMPLES)
print("======================================")


MILE -- pSMILE/NOISYSMILE MINI-BATCH-TEST (02c)
Run name:               mile_psmile_k3_prior0.02_mb32_spc60_thin30_testeval
Chain seeds:            [1, 2, 3]
Prior std:              0.02  (Autoren-Default: Standard-Gaussian)
Minibatch size:         32
Initial step-size grid: [0.0001, 0.0002, 0.0003, 0.0005, 0.0007, 0.0009, 0.001]
num_integration_steps:  1
reset_prob:             0.99  (Library-Default)
adaption_prob:          0.05  (Library-Default)
adaption_strength:      0.02  (Library-Default)
smoothing_factor:       0.01  (Library-Default)
N_SAMPLES_PER_CHAIN:    60  (KLEIN -- Smoke-Test)
THINNING_STEPS:         30  (KLEIN -- Smoke-Test)
BURN_IN_SAMPLES:        0


In [2]:
# ============================================================
# SMILE-REPO KLONEN (falls noch nicht vorhanden, z.B. neue Session) + IMPORTS
# ============================================================

import sys
import subprocess
import runpy
import time

import numpy as np
import jax
import jax.numpy as jnp
from jax import random

if not SMILE_REPO.exists():
    print(f"SMILE-Repo nicht gefunden unter {SMILE_REPO} -- klone es jetzt...")
    result = subprocess.run(
        ["git", "clone", "https://github.com/EmanuelSommer/SMILE", str(SMILE_REPO)],
        capture_output=True, text=True,
    )
    print(result.stdout[-1500:])
    if result.returncode != 0:
        print(result.stderr[-1500:])
        raise RuntimeError("git clone von SMILE fehlgeschlagen -- siehe Fehlermeldung oben.")
    print("SMILE-Repo geklont:", SMILE_REPO)
else:
    print("SMILE-Repo bereits vorhanden:", SMILE_REPO)

os.chdir(QWEN_REPO)
if str(QWEN_REPO) not in sys.path:
    sys.path.insert(0, str(QWEN_REPO))
if str(SMILE_REPO) not in sys.path:
    sys.path.insert(0, str(SMILE_REPO))

print("Python:", sys.executable)
print("JAX:", jax.__version__)
print("Devices:", jax.devices())

assert QWEN_REPO.exists()
assert BASELINE_SCRIPT.exists()
assert SMILE_REPO.exists()


SMILE-Repo bereits vorhanden: /dss/dsshome1/00/ra58vit2/Masterarbeit/SMILE
Python: /dss/dsshome1/00/ra58vit2/Masterarbeit/bayes_sub_inf/.venv/bin/python3
JAX: 0.5.3


Devices: [CudaDevice(id=0)]


In [3]:
# ============================================================
# NOISYSMILE-KERNEL AUS DEM SMILE-REPO IMPORTIEREN ("pSMILE")
# ============================================================

try:
    from src.kernels.smile import NOISYSMILE
except ImportError as exc:
    raise ImportError(
        "Konnte 'src.kernels.smile.NOISYSMILE' nicht importieren. Pruefe, ob SMILE_REPO "
        "korrekt geklont wurde und die src/-Struktur wie im GitHub-Repo aussieht. "
        f"Original-Fehler: {exc}"
    )

print("NOISYSMILE-Kernel (pSMILE) erfolgreich importiert:", NOISYSMILE)


NOISYSMILE-Kernel (pSMILE) erfolgreich importiert: <class 'src.kernels.smile.NOISYSMILE'>


In [4]:
# ============================================================
# GETEILTES MODUL (unveraendert -- nur bestehende Bausteine wiederverwendet)
# ============================================================

import qwen_posterior_utils as qpu

qpu.patch_subspace_curve_predict()


Patched SubspaceBaseModel._predict: lax.cond bypass for concrete train flags active.


In [5]:
# ============================================================
# LOAD QWEN AG-NEWS BASELINE (env/data + LoRA-Struktur)
# ============================================================

baseline_objects = runpy.run_path(str(BASELINE_SCRIPT))

env = baseline_objects["env"]
params = baseline_objects["params"]
data = baseline_objects["data"]
rng_key = baseline_objects["rng_key"]

print("Baseline geladen. Model:", type(env.s_model))


Dataset loaded: 120000 samples, 4 classes
Target token IDs: [[15]
 [16]
 [17]
 [18]]


Validation data are used
Training data: 108000 samples, Validation data: 12000 samples
Computed cp_fix: [1]
Setting num_epochs to 1 to match 500 steps


Using cosine rho schedule with total steps 500
save best parameters
Loaded checkpoint: tmp_files/955nvvay_pretrained_params.npy
Saved object shape: (1,)


post_predict_logits shape: (12000, 4)


post_predict_logits shape: (12000, 4)


post_predict_logits shape: (7600, 4)


post_predict_logits shape: (7600, 4)

SAVED BEST CHECKPOINT - VALIDATION
val_ll: -0.246206
val_acc: 0.913667
val_ece: 0.009223
val_brier: 0.131583
val_mutual_information: 0.000000
val_mean_entropy: 0.000000

SAVED BEST CHECKPOINT - TEST
test_ll: -0.251105
test_acc: 0.914737
test_ece: 0.007043
test_brier: 0.131577
test_mutual_information: 0.000000
test_mean_entropy: 0.000000
Baseline geladen. Model: <class 'subspace_inference.curve_optimizer.subspace_curve.LoRACategorySubspace'>


In [6]:
# ============================================================
# VOLLER TRAININGSSATZ (108.000 Beispiele) fuer Mini-Batch-Sampling
# ============================================================

train_inputs_full, train_labels_full = data.get("train")
N_TOTAL_TRAIN = int(train_labels_full.shape[0])
print("Voller Trainingssatz:", N_TOTAL_TRAIN, "Beispiele")


def sample_minibatch(rng_key):
    rng_key, batch_key = random.split(rng_key)
    batch_indices = random.randint(
        batch_key, shape=(MINIBATCH_SIZE,), minval=0, maxval=N_TOTAL_TRAIN,
    )
    x_batch = {
        key: value[batch_indices, -SEQ_LEN:]
        for key, value in train_inputs_full.items()
    }
    y_batch = train_labels_full[batch_indices]
    return x_batch, y_batch, rng_key


Voller Trainingssatz: 108000 Beispiele


In [7]:
# ============================================================
# LORA-VEKTORISIERUNG + MINI-BATCH LOG-POSTERIOR + GRAD-ESTIMATOR
# (identisch zu 02b -- gleiche grad_estimator-Signatur, die NOISYSMILE
# genauso erwartet: grad_estimator(position, x=..., y=...) -> (value, grad))
# ============================================================

_, _, _, qwen_subset_nll_sum = qpu.build_qwen_nll_fns(env)

theta_map, all_leaves, lora_indices, full_tree_definition, unravel_lora = (
    qpu.vectorize_lora_params(params["params"])
)
rebuild_full_params = qpu.build_rebuild_fn(
    all_leaves, lora_indices, full_tree_definition, unravel_lora,
)

NLL_KEY_DUMMY = random.PRNGKey(0)  # lora_rho_w=lora_rho_s=0 -> key wirkt sich nicht aus


def qwen_log_posterior_minibatch(theta, x_batch, y_batch, n_total):
    candidate_params = rebuild_full_params(theta)
    batch_keys = random.split(NLL_KEY_DUMMY, y_batch.shape[0])

    nll_sum_batch, _ = qwen_subset_nll_sum(
        candidate_params, x_batch, y_batch, batch_keys,
        batch_size=NLL_INNER_BATCH_SIZE,
    )
    batch_size = y_batch.shape[0]
    nll_estimate_full = (nll_sum_batch / batch_size) * n_total

    theta_float32 = theta.astype(jnp.float32)
    squared_norm = jnp.sum(jnp.square(theta_float32))

    log_likelihood_estimate = -nll_estimate_full
    log_prior = -0.5 * squared_norm / jnp.asarray(PRIOR_STD**2, dtype=jnp.float32)

    return log_likelihood_estimate + log_prior


def grad_estimator(position, x, y):
    # NOISYSMILE-Interface: (value, grad) = grad_estimator(position, x=.., y=..).
    def scalar_fn(theta_):
        return qwen_log_posterior_minibatch(theta_, x, y, N_TOTAL_TRAIN)
    return jax.value_and_grad(scalar_fn)(position)


print("theta_map shape:", theta_map.shape)
print("grad_estimator bereit.")


LoRA objects: 48
Sampling vector shape: (540672,)
Sampling parameters: 540672


All MAP parameters finite: True
theta_map shape: (540672,)
grad_estimator bereit.


In [8]:
# ============================================================
# CHAIN-STARTPUNKTE LADEN (identisch zu 02b)
# ============================================================

_eval_script_template = BASELINE_SCRIPT.read_text(encoding="utf-8")


def load_chain_theta_map(seed):
    seed_dir = CHECKPOINTS_ROOT / f"qwen_lora_agnews_seed{seed}"
    checkpoint_path = list(seed_dir.glob("*_pretrained_params.npy"))[0]
    old_checkpoint_line = next(
        line for line in _eval_script_template.splitlines()
        if line.strip().startswith("CHECKPOINT_PATH")
    )
    old_seed_line = next(
        line for line in _eval_script_template.splitlines()
        if line.strip().startswith("SEED")
    )
    script_text = _eval_script_template.replace(
        old_checkpoint_line, f'CHECKPOINT_PATH = r"{checkpoint_path}"',
    ).replace(old_seed_line, f"SEED = {seed}")
    tmp_script_path = (
        QWEN_REPO / "experiments" / "ag_news_qwen_lora" / f"_tmp_eval_chain_seed{seed}_02c.py"
    )
    tmp_script_path.write_text(script_text, encoding="utf-8")
    try:
        chain_objects = runpy.run_path(str(tmp_script_path))
    finally:
        tmp_script_path.unlink(missing_ok=True)
    theta, *_ = qpu.vectorize_lora_params(chain_objects["params"]["params"])
    return theta


chain_theta_maps = {seed: load_chain_theta_map(seed) for seed in CHAIN_SEEDS}
print("Chain-Startpunkte geladen:", list(chain_theta_maps.keys()))


Dataset loaded: 120000 samples, 4 classes
Target token IDs: [[15]
 [16]
 [17]
 [18]]
Validation data are used
Training data: 108000 samples, Validation data: 12000 samples
Computed cp_fix: [1]
Setting num_epochs to 1 to match 500 steps


Using cosine rho schedule with total steps 500
save best parameters
Loaded checkpoint: /dss/dsshome1/00/ra58vit2/Masterarbeit/checkpoints/qwen_lora_agnews_seed1/iiljya0g_pretrained_params.npy
Saved object shape: (1,)


post_predict_logits shape: (12000, 4)


post_predict_logits shape: (12000, 4)


post_predict_logits shape: (7600, 4)


...

post_predict_logits shape: (12000, 4)


post_predict_logits shape: (12000, 4)


post_predict_logits shape: (7600, 4)


post_predict_logits shape: (7600, 4)

SAVED BEST CHECKPOINT - VALIDATION
val_ll: -0.244713
val_acc: 0.916167
val_ece: 0.006295
val_brier: 0.128253
val_mutual_information: 0.000000
val_mean_entropy: 0.000000

SAVED BEST CHECKPOINT - TEST
test_ll: -0.258215
test_acc: 0.911053
test_ece: 0.009402
test_brier: 0.134708
test_mutual_information: 0.000000
test_mean_entropy: 0.000000
LoRA objects: 48
Sampling vector shape: (540672,)
Sampling parameters: 540672
All MAP parameters finite: True


Chain-Startpunkte geladen: [1, 2, 3]


In [9]:
# ============================================================
# NOISYSMILE OHNE PMAP INSTANZIIEREN (Single-GPU-Bypass, wie bei SGHMC in 02b).
# NOISYSMILE.__init__ wuerde selbst _compile_sample_step(**kwargs) aufrufen
# (jax.pmap-Wrapper fuer Multi-GPU) -- das umgehen wir bewusst.
# ============================================================

psmile_obj = object.__new__(NOISYSMILE)
psmile_obj._grad_estimator = grad_estimator

print("NOISYSMILE-Objekt (ohne pmap) bereit:", psmile_obj)


NOISYSMILE-Objekt (ohne pmap) bereit: <src.kernels.smile.NOISYSMILE object at 0x7273c0544610>


In [10]:
# ============================================================
# RUN_ONE_CHAIN_PSMILE -- jax.jit von Anfang an eingebaut (Lehre aus 02b:
# ungejitteter Aufruf von _sample_step wird bei JEDEM Schritt neu getraced,
# ~25min/Sample statt Sekunden). EINE Kompilierung pro Chain, dann schnelle
# Wiederverwendung -- exakt das Muster aus Sampler._compile_sample_step in
# der Bibliothek selbst, nur ohne deren jax.pmap-Wrapper.
#
# WICHTIG (siehe Intro-Zelle): der `step_size`-Parameter von _sample_step
# ist bei NOISYSMILE FUNKTIONSLOS -- massgeblich ist state.step_size (siehe
# psmile-instance-Zelle unten, State-Konstruktion). Deshalb wird step_size
# hier NICHT in functools.partial eingebrannt.
# ============================================================

import functools


def build_jitted_psmile_step(num_integration_steps, reset_prob, adaption_prob,
                              adaption_strength, smoothing_factor):
    # Baut EINMAL eine jax.jit-kompilierte Sample-Step-Funktion mit fest
    # eingebrannten (nicht getracten) Hyperparametern -- ausser step_size,
    # der bei NOISYSMILE ohnehin ignoriert wird (siehe oben).
    partial_step = functools.partial(
        NOISYSMILE._sample_step,
        psmile_obj,
        num_integration_steps=num_integration_steps,
        reset_prob=reset_prob,
        adaption_prob=adaption_prob,
        adaption_strength=adaption_strength,
        smoothing_factor=smoothing_factor,
    )
    return jax.jit(partial_step)


def run_one_chain_psmile(chain_theta_map, rng_key, initial_step_size, chain_label):
    # initial_step_size geht hier rein -- beim State-Bauen, nicht bei _sample_step.
    #
    # WICHTIG (Fix nach echtem Fehler auf dem Cluster): NOISYSMILE.State.__init__ tiled
    # step_size/logdensity/energy_var_mean/energy_var_std/time normalerweise auf
    # jax.tree.leaves(position)[0].shape[0] -- das ist fuer den Bibliotheks-Normalfall
    # gedacht, wo `position` eine FUEHRENDE CHAIN-ACHSE hat (jax.pmap ueber mehrere
    # Chains) und diese Felder "ein Skalar pro Chain" sein sollen. Unser `position` ist
    # aber ein flacher 540.672-Parameter-Vektor OHNE Chain-Achse -- der Code haette das
    # faelschlich als "540.672 Chains" interpretiert und alle fuenf Felder auf Shape
    # (540672,) statt Skalar () aufgeblasen (siehe TypeError: state.logdensity hatte
    # float32[540672] statt float32[], scan-Carry-Shape-Mismatch).
    #
    # Fix: diese fuenf Felder explizit als JAX-Skalare (nicht Python float) uebergeben --
    # dann greift im Bibliothekscode der `else`-Zweig, der sie unveraendert uebernimmt
    # statt zu tilen (siehe isinstance(step_size, float)-Check in State.__init__).
    state = NOISYSMILE.State(
        position=chain_theta_map,
        step_size=jnp.asarray(initial_step_size, dtype=jnp.float32),
        logdensity=jnp.asarray(0.0, dtype=jnp.float32),
        energy_var_mean=jnp.asarray(0.0, dtype=jnp.float32),
        energy_var_std=jnp.asarray(0.0, dtype=jnp.float32),
        time=jnp.asarray(1.0, dtype=jnp.float32),
    )
    collected_samples = []
    collected_step_sizes = []

    jitted_step = build_jitted_psmile_step(
        num_integration_steps=NUM_INTEGRATION_STEPS,
        reset_prob=RESET_PROB,
        adaption_prob=ADAPTION_PROB,
        adaption_strength=ADAPTION_STRENGTH,
        smoothing_factor=SMOOTHING_FACTOR,
    )

    for sample_index in range(N_SAMPLES_PER_CHAIN):
        sample_start_time = time.time()

        for _ in range(THINNING_STEPS):
            rng_key, step_key = random.split(rng_key)
            x_batch, y_batch, rng_key = sample_minibatch(rng_key)

            state = jitted_step(state, step_key, (x_batch, y_batch))

        sample_duration = time.time() - sample_start_time

        position_host = np.asarray(jax.device_get(state.position))
        if not np.isfinite(position_host).all():
            print(f"[{chain_label}] Sample {sample_index+1}/{N_SAMPLES_PER_CHAIN}: NICHT endlich -- abgebrochen.")
            return None, None, rng_key

        distance = float(np.linalg.norm(position_host - np.asarray(chain_theta_map)))
        mean_step_size = float(jnp.mean(state.step_size))
        print(
            f"[{chain_label}] Sample {sample_index+1}/{N_SAMPLES_PER_CHAIN}: "
            f"endlich, Distanz vom Start={distance:.4f}, mean(step_size)={mean_step_size:.6g}, "
            f"Dauer={sample_duration:.1f}s"
        )
        collected_samples.append(position_host)
        collected_step_sizes.append(mean_step_size)

    sample_positions = np.stack(collected_samples, axis=0)
    return sample_positions, collected_step_sizes, rng_key


In [11]:
# ============================================================
# STEP-SIZE-GRID-TEST: alle Werte in STEP_SIZE_GRID nacheinander testen,
# jeweils ueber alle CHAIN_SEEDS. Nur Sampling, keine Auswertung -- Ziel
# ist der Distanz-Verlauf pro step_size, um begruendet einen Wert fuer den
# finalen Produktionslauf auszuwaehlen (SMILE-Paper-Tuningrezept: grobes
# Grid ueber mehrere Groessenordnungen, dann Wert auswaehlen).
# ============================================================

import gc

print(f"Step-Size-Grid: {STEP_SIZE_GRID}")
print(f"Chains: {CHAIN_SEEDS}   N_SAMPLES_PER_CHAIN={N_SAMPLES_PER_CHAIN}   THINNING_STEPS={THINNING_STEPS}")

grid_results = {}

for step_size in STEP_SIZE_GRID:
    print()
    print("======================================")
    print(f"STEP_SIZE = {step_size}")
    print("======================================")

    chain_results = {}
    chain_step_size_traces = {}

    for seed in CHAIN_SEEDS:
        print(f"\n=== step_size={step_size}  Chain seed={seed} ===")
        local_rng = random.PRNGKey(4242 + seed)
        samples, step_sizes, local_rng = run_one_chain_psmile(
            chain_theta_maps[seed], local_rng, step_size, f"step={step_size} seed={seed}",
        )
        chain_results[seed] = samples
        chain_step_size_traces[seed] = step_sizes

        gc.collect()
        jax.clear_caches()

    grid_results[step_size] = (chain_results, chain_step_size_traces)

print()
print("======================================")
print("GESAMTZUSAMMENFASSUNG STEP-SIZE-GRID")
print("======================================")
for step_size in STEP_SIZE_GRID:
    chain_results, chain_step_size_traces = grid_results[step_size]
    print(f"\n--- step_size={step_size} ---")
    for seed in CHAIN_SEEDS:
        samples = chain_results[seed]
        if samples is None:
            print(f"  seed={seed}: FEHLGESCHLAGEN (NaN/Inf)")
            continue
        final_distance = float(np.linalg.norm(
            np.asarray(samples[-1]) - np.asarray(chain_theta_maps[seed])
        ))
        trace = chain_step_size_traces[seed]
        print(
            f"  seed={seed}: OK, finale Distanz={final_distance:.4f}, "
            f"mean(step_size) {trace[0]:.6g} -> {trace[-1]:.6g}"
        )


Step-Size-Grid: [0.0001, 0.0002, 0.0003, 0.0005, 0.0007, 0.0009, 0.001]
Chains: [1, 2, 3]   N_SAMPLES_PER_CHAIN=60   THINNING_STEPS=30

STEP_SIZE = 0.0001

=== step_size=0.0001  Chain seed=1 ===


[step=0.0001 seed=1] Sample 1/60: endlich, Distanz vom Start=0.0027, mean(step_size)=0.000121899, Dauer=251.0s


[step=0.0001 seed=1] Sample 2/60: endlich, Distanz vom Start=0.0062, mean(step_size)=0.000142825, Dauer=2.0s


[step=0.0001 seed=1] Sample 3/60: endlich, Distanz vom Start=0.0101, mean(step_size)=0.000151567, Dauer=2.0s


[step=0.0001 seed=1] Sample 4/60: endlich, Distanz vom Start=0.0145, mean(step_size)=0.000164061, Dauer=2.0s


...

[step=0.001 seed=3] Sample 57/60: endlich, Distanz vom Start=9.6890, mean(step_size)=0.000937245, Dauer=2.0s


[step=0.001 seed=3] Sample 58/60: endlich, Distanz vom Start=9.7553, mean(step_size)=0.000882127, Dauer=2.0s


[step=0.001 seed=3] Sample 59/60: endlich, Distanz vom Start=9.7270, mean(step_size)=0.000917765, Dauer=2.0s


[step=0.001 seed=3] Sample 60/60: endlich, Distanz vom Start=9.7244, mean(step_size)=0.000954843, Dauer=2.0s



GESAMTZUSAMMENFASSUNG STEP-SIZE-GRID

--- step_size=0.0001 ---
  seed=1: OK, finale Distanz=3.1888, mean(step_size) 0.000121899 -> 0.000321733
  seed=2: OK, finale Distanz=1.4462, mean(step_size) 8.68126e-05 -> 8.65524e-05
  seed=3: OK, finale Distanz=1.7197, mean(step_size) 9.03921e-05 -> 0.000112054

--- step_size=0.0002 ---
  seed=1: OK, finale Distanz=5.8993, mean(step_size) 0.000243799 -> 0.000618975
  seed=2: OK, finale Distanz=2.5710, mean(step_size) 0.000173625 -> 0.000187299
  seed=3: OK, finale Distanz=3.1343, mean(step_size) 0.000180784 -> 0.000224288

--- step_size=0.0003 ---
  seed=1: OK, finale Distanz=8.6855, mean(step_size) 0.000365698 -> 0.000823459
  seed=2: OK, finale Distanz=3.3648, mean(step_size) 0.000260438 -> 0.000298623
  seed=3: OK, finale Distanz=4.2891, mean(step_size) 0.000271176 -> 0.000323239

--- step_size=0.0005 ---
  seed=1: OK, finale Distanz=11.1645, mean(step_size) 0.000609497 -> 0.00137849
  seed=2: OK, finale Distanz=7.1798, mean(step_size) 0.000